# Búsqueda de hiperparámetros

`03_modeling.ipynb` eligió XGBoost sin reponderar las clases —sin `scale_pos_weight` ni remuestreo, algo
que ahí se midió antes que suponerse— y con el umbral 0,2: **+128.701** sobre aprobar todo en los cuatro
bloques de validación, contra **+23.999** de la mejor regla de un corte sobre `score`. Pero corrió con
una configuración razonable y sin ajustar.

Esta notebook busca una mejor, y **no toca el período reservado**. Eso no es una promesa escrita al
margen: esta notebook ni siquiera lo carga. Lo que produce es un archivo de configuración que
`05_evaluation.ipynb` levanta para entrenar el modelo final. El período reservado se usa una sola vez,
allá, cuando ya no queda nada por elegir.

## 1. Qué se optimiza y con qué

**Qué se optimiza: pesos.** El objetivo es la ganancia sobre aprobar todo, sumada en los cuatro folds,
con el umbral 0,2. No el AUC. La sección 2 de `03_modeling.ipynb` mostró por qué importa la distinción:
entre la logística y XGBoost hay 0,04 de AUC y apenas un 1% de diferencia en plata. Optimizar AUC sería
optimizar una métrica que se correlaciona con el objetivo pero no es el objetivo.

**Con qué se valida: los mismos folds temporales** de `temporal_folds`, con el mismo `N_SPLITS`. Un
`KFold` aleatorio dejaría entrenar con transacciones posteriores a las de validación.

**Por qué Optuna y no una grilla.** El espacio tiene ocho dimensiones, la mayoría continuas. Una grilla
de apenas tres valores por dimensión son 6.561 combinaciones, y a unos cinco segundos cada una no
termina nunca. Optuna aporta dos cosas concretas:

- **TPE** en lugar de azar: modela qué regiones del espacio vinieron dando buenos resultados y muestrea
  ahí, en vez de probar a ciegas como `RandomizedSearchCV`.
- **Pruning**: como la ganancia se acumula fold por fold, una prueba que arranca mal se corta después del
  primer fold en vez de gastar los cuatro.

**Dónde quedan registradas.** Cada prueba se guarda en MLflow con sus hiperparámetros y su ganancia, y la
búsqueda entera queda como una corrida madre. Es lo que permite volver a mirar por qué se eligió esta
configuración sin depender de que el notebook siga ejecutado.

**Hipótesis, y es la que importa para el informe.** La búsqueda va a mejorar la ganancia en validación
—siempre lo hace: es un máximo sobre 40 intentos—, pero **poco de esa mejora va a sobrevivir fuera de
muestra**. Es el mismo mecanismo que con el umbral en `03`: elegir mirando los mismos cuatro folds donde
después se reporta convierte parte del ruido en ganancia aparente. Si la hipótesis es correcta, la
ventaja del modelo ajustado sobre el de defecto tiene que encogerse bastante cuando `05` lo mida sobre
datos nuevos.

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from fraud_detection.constants import DATE, N_SPLITS, RANDOM_STATE, TARGET
from fraud_detection.dataset import load_transactions, normalize_text, split_development
from fraud_detection.evaluation import expected_gain, optimal_probability_threshold
from fraud_detection.features import build_preprocessor
from fraud_detection.modeling import temporal_folds
from fraud_detection.tracking import configurar, mlflow, registrar

optuna.logging.set_verbosity(optuna.logging.WARNING)

development, _ = split_development(normalize_text(load_transactions()))
X, y = development.drop(columns=[TARGET]), development[TARGET]
folds = list(temporal_folds(development[DATE], n_splits=N_SPLITS))
UMBRAL = optimal_probability_threshold()
NOMBRES = [f"fold {numero}" for numero in range(len(folds))]
ARTEFACTOS = Path("../models")
ARTEFACTOS.mkdir(exist_ok=True)

BASE = dict(tree_method="hist", eval_metric="aucpr", n_jobs=-1, random_state=RANDOM_STATE)
DEFECTO = dict(
    n_estimators=500, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, **BASE
)
print("store de experimentos:", configurar("04-busqueda-hiperparametros"))


def modelo_xgb(parametros: dict) -> Pipeline:
    """Pipeline completo con los hiperparámetros dados."""
    return Pipeline(
        [("preprocesamiento", build_preprocessor()), ("modelo", XGBClassifier(**parametros))]
    )


def ganancia_en_folds(parametros: dict, trial=None) -> float:
    """Ganancia sobre aprobar todo, sumada en los folds temporales."""
    total = 0.0
    for numero, (train_pos, valid_pos) in enumerate(folds):
        pipeline = modelo_xgb(parametros).fit(X.iloc[train_pos], y.iloc[train_pos])
        valid = development.iloc[valid_pos]
        probabilidad = pd.Series(pipeline.predict_proba(X.iloc[valid_pos])[:, 1], index=valid.index)
        piso = expected_gain(valid, pd.Series(True, index=valid.index))
        total += expected_gain(valid, probabilidad.lt(UMBRAL)) - piso
        if trial is not None:
            trial.report(total, numero)
            if trial.should_prune():
                raise optuna.TrialPruned()
    return total


def objetivo(trial) -> float:
    """Espacio de búsqueda de XGBoost, evaluado en ganancia."""
    return ganancia_en_folds(
        dict(
            n_estimators=trial.suggest_int("n_estimators", 200, 900, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            max_depth=trial.suggest_int("max_depth", 3, 8),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 20, log=True),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            reg_lambda=trial.suggest_float("reg_lambda", 0.01, 10.0, log=True),
            gamma=trial.suggest_float("gamma", 0.0, 5.0),
            **BASE,
        ),
        trial=trial,
    )


def anotar_prueba(estudio, prueba):
    """Guardar cada prueba completada como una corrida anidada en MLflow."""
    if prueba.value is not None:
        registrar(
            f"prueba {prueba.number}",
            parametros=prueba.params,
            metricas={"ganancia": prueba.value},
            anidada=True,
        )


inicio = time.time()
estudio = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1),
)

with mlflow.start_run(run_name="busqueda con TPE"):
    estudio.optimize(objetivo, n_trials=40, callbacks=[anotar_prueba])
    ganancia_defecto = ganancia_en_folds(DEFECTO)
    podadas = sum(prueba.state == optuna.trial.TrialState.PRUNED for prueba in estudio.trials)
    pruebas = estudio.trials_dataframe()
    pruebas.to_csv(ARTEFACTOS / "pruebas_optuna.csv", index=False)
    mlflow.log_params(estudio.best_params)
    mlflow.log_metrics(
        {
            "ganancia_ajustado": estudio.best_value,
            "ganancia_defecto": ganancia_defecto,
            "pruebas_podadas": podadas,
        }
    )
    mlflow.log_artifact(str(ARTEFACTOS / "pruebas_optuna.csv"))

print(f"{len(estudio.trials)} pruebas en {time.time() - inicio:.0f}s, {podadas} podadas")
print(f"defecto:  {ganancia_defecto:,.0f}")
print(f"ajustado: {estudio.best_value:,.0f}  ({estudio.best_value - ganancia_defecto:+,.0f})")
display(pd.Series(estudio.best_params).to_frame("mejor valor"))

store de experimentos: sqlite:////Users/matiasmoyano/repositories/mercadolibre-fraud-detection/mlflow.db


40 pruebas en 94s, 21 podadas
defecto:  128,701
ajustado: 133,828  (+5,127)


,mejor valor
n_estimators,400.000000
learning_rate,0.123289
max_depth,3.000000
min_child_weight,1.000000
subsample,0.869562
colsample_bytree,0.668390
reg_lambda,0.032014
gamma,1.486420


**Lectura: la búsqueda mejora, y el pruning hace que salga barata.**

De las 40 pruebas, **21 se podaron** después del primer o segundo fold, así que el costo real fue
bastante menor que 40 ajustes completos. La mejor configuración rinde **+133.828** contra los +128.701
del modelo de defecto: **+5.127**, un 4% más.

El sentido del ajuste es informativo: la búsqueda se fue hacia **menos capacidad y más regularización
por árbol** —`max_depth` 3 en vez de 5, `gamma` 1,49, 400 árboles con un `learning_rate` más alto—, que
es hacia dónde suele ir cuando los bloques de entrenamiento son cortos. El fold 0 entrena con 8 días.

Eso sí, +5.127 elegidos sobre los mismos cuatro folds donde se los mide no son +5.127 comprobados.

## 2. ¿La mejora es pareja entre folds?

Antes de dar la configuración por buena hay una pregunta barata de responder. Si el ajuste captó algo
real del problema, tendría que mejorar en **todos** los folds. Si mejora mucho en uno y empeora en otro,
lo que capturó es la particularidad de un período.

In [2]:
AJUSTADO = {**estudio.best_params, **BASE}

comparacion = []
for numero, (train_pos, valid_pos) in enumerate(folds):
    valid = development.iloc[valid_pos]
    piso = expected_gain(valid, pd.Series(True, index=valid.index))
    fila = {}
    for nombre, parametros in [("defecto", DEFECTO), ("ajustado", AJUSTADO)]:
        pipeline = modelo_xgb(parametros).fit(X.iloc[train_pos], y.iloc[train_pos])
        probabilidad = pd.Series(pipeline.predict_proba(X.iloc[valid_pos])[:, 1], index=valid.index)
        fila[f"g_{nombre}"] = expected_gain(valid, probabilidad.lt(UMBRAL)) - piso
        fila[f"auc_{nombre}"] = roc_auc_score(valid[TARGET], probabilidad)
    fila["diferencia"] = fila["g_ajustado"] - fila["g_defecto"]
    comparacion.append(fila)

comparacion = pd.DataFrame(comparacion, index=NOMBRES)
comparacion.loc["total"] = comparacion.sum()
comparacion.loc["total", ["auc_defecto", "auc_ajustado"]] = np.nan
display(comparacion.round(4))

for nombre, parametros in [("defecto", DEFECTO), ("ajustado", AJUSTADO)]:
    registrar(
        f"validacion {nombre}",
        parametros=parametros,
        metricas={
            "ganancia": comparacion.loc["total", f"g_{nombre}"],
            "auc_medio": comparacion[f"auc_{nombre}"].iloc[:-1].mean(),
        },
        etiquetas={"etapa": "validacion temporal"},
    )

,g_defecto,auc_defecto,g_ajustado,auc_ajustado,diferencia
fold 0,27765.885,0.8795,29986.7225,0.8824,2220.8375
fold 1,30049.290,0.8840,33345.8750,0.8848,3296.5850
fold 2,32131.810,0.8819,31493.4000,0.8855,-638.4100
fold 3,38754.155,0.8885,39002.2525,0.8874,248.0975
total,128701.140,NaN,133828.2500,NaN,5127.1100


**Lectura: mejora en tres de cuatro, no en los cuatro.**

Los folds 0, 1 y 3 mejoran —**+2.221**, **+3.297** y **+248**— y el fold 2 **empeora 638**. La mejora
tampoco se reparte: entre los folds 0 y 1 suman 5.517 de los 5.127 netos, y el fold 3 aporta 248.

Hay un detalle que refuerza lo que ya venía apareciendo: en el fold 2 el modelo ajustado tiene **mejor
AUC** (0,8855 contra 0,8819) y aun así **pierde plata**. Ordena mejor y decide peor, porque la decisión
se toma con un único corte.

Nada de esto alcanza para descartar el ajuste —el saldo es positivo y el AUC mejora en tres de los
cuatro folds—, pero sí para desconfiar del tamaño de la mejora, que es lo que predecía la hipótesis.
**Solo el período reservado lo resuelve, y eso pasa en `05_evaluation.ipynb`.**

## 3. Lo que se entrega

**Se declara acá, antes de medir nada nuevo: el modelo que va al período reservado es el ajustado.** El
de defecto se va a reportar al lado como referencia, no como una alternativa a elegir después de ver los
resultados. Dejarlo escrito antes es lo que hace que esa comparación signifique algo.

La configuración se guarda en `models/mejores_parametros.json`, que es lo que carga `05`. Junto con el
registro en MLflow, eso alcanza para reconstruir por qué se eligió: los hiperparámetros, la ganancia en
validación, y las 40 pruebas con sus valores.

In [3]:
configuracion = {
    "parametros": AJUSTADO,
    "ganancia_validacion": estudio.best_value,
    "ganancia_validacion_defecto": ganancia_defecto,
    "umbral": UMBRAL,
    "pruebas": len(estudio.trials),
    "folds": N_SPLITS,
}
destino = ARTEFACTOS / "mejores_parametros.json"
destino.write_text(json.dumps(configuracion, indent=2, ensure_ascii=False))

print(destino.read_text())
print(
    "corridas registradas:",
    len(mlflow.search_runs(experiment_names=["04-busqueda-hiperparametros"])),
)

{
  "parametros": {
    "n_estimators": 400,
    "learning_rate": 0.12328884725132523,
    "max_depth": 3,
    "min_child_weight": 1,
    "subsample": 0.8695623984396229,
    "colsample_bytree": 0.6683897041284035,
    "reg_lambda": 0.03201406263276564,
    "gamma": 1.4864197693847574,
    "tree_method": "hist",
    "eval_metric": "aucpr",
    "n_jobs": -1,
    "random_state": 42
  },
  "ganancia_validacion": 133828.25,
  "ganancia_validacion_defecto": 128701.14000000003,
  "umbral": 0.2,
  "pruebas": 40,
  "folds": 4
}
corridas registradas: 43


Con eso, `05_evaluation.ipynb` puede entrenar el modelo final sin volver a buscar nada, y el período
reservado se toca una sola vez.

Para revisar las corridas:

```bash
uv run --locked mlflow ui --backend-store-uri sqlite:///mlflow.db
```